In [ ]:
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BartTokenizerFast, BartForSequenceClassification
import torch
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import recall_score, f1_score, roc_auc_score
from scipy.special import softmax
from transformers import Trainer, TrainingArguments
import wandb


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


환경 준비

In [ ]:
# !pip install transformers datasets pandas scikit-learn wandb --quiet

모델 및 데이터셋 설정

In [ ]:
# 🔹 CSV 파일 불러오기
df = pd.read_csv("/content/drive/MyDrive/디스부 최종 프로젝트(온라인 그루밍 범죄 탐지)/final_2.csv", encoding = 'cp949')  # text, label 열 포함
df

,text,label
0,<SPEAKER_A>안녕! <SPEAKER_B>안녕! <SPEAKER_A>잘 지냈어...,1
1,<SPEAKER_A>이번 주말에 어때? <SPEAKER_A>달콤한 사람아 <SPEA...,1
2,<SPEAKER_A>안녕! <SPEAKER_A>안녕! <SPEAKER_A>좀 늦었는...,1
3,<SPEAKER_A>안녕! <SPEAKER_B>안녕! <SPEAKER_B>어디 갔었...,1
4,"<SPEAKER_A>안녕, 섹시! <SPEAKER_A>잘 지냈어? <SPEAKER_...",1
...,...,...
1833,"<SPEAKER_A>헤이,<SPEAKER_B>헤이야!<SPEAKER_A>너는 어때?...",0
1834,<SPEAKER_A>안녕<SPEAKER_B>안녕<SPEAKER_B>??<SPEAKE...,0
1835,<SPEAKER_A>안녕!<SPEAKER_B>안녕<SPEAKER_A>무슨 일이야?<...,0
1836,<SPEAKER_A>인터페이스에 HasProperty가 NamedGetter가 찾을...,0


In [ ]:
# 🔹 Train/Val 나누기
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

# 🔹 Tokenizer
tokenizer = BartTokenizerFast.from_pretrained("gogamza/kobart-base-v2")

# 🔹 Custom Dataset
class GroomingDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

        # 스페셜 토큰 ID 저장
        self.speaker_tokens = [
            self.tokenizer.convert_tokens_to_ids('<SPEAKER_A>'),
            self.tokenizer.convert_tokens_to_ids('<SPEAKER_B>')
        ]

    def __len__(self):
        return len(self.texts)

    def find_valid_start(self, input_ids):
        # 뒤에서 max_len만큼 자르고, 해당 범위 내에서 가장 마지막 <SPEAKER_*> 토큰을 찾음
        start_idx = max(0, len(input_ids) - self.max_len)
        candidate = input_ids[start_idx:]

        for i in range(len(candidate)):
            if candidate[i] in self.speaker_tokens:
                return start_idx + i

        # 못 찾으면 그냥 max_len만큼 자른다
        return len(input_ids) - self.max_len

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoded = self.tokenizer(
            text,
            add_special_tokens=False,
            return_attention_mask=False
        )
        input_ids = encoded["input_ids"]

        # valid_start를 찾아서 슬라이싱
        if len(input_ids) > self.max_len:
            valid_start = self.find_valid_start(input_ids)
            input_ids = input_ids[valid_start:]

        # special tokens 추가
        input_ids = self.tokenizer.build_inputs_with_special_tokens(input_ids)

        # attention mask 및 padding
        attention_mask = [1] * len(input_ids)
        padding_len = self.max_len + 2 - len(input_ids)

        input_ids += [self.tokenizer.pad_token_id] * padding_len
        attention_mask += [0] * padding_len

        return {
            'input_ids': torch.tensor(input_ids),
            'attention_mask': torch.tensor(attention_mask),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = GroomingDataset(train_df, tokenizer)
val_dataset = GroomingDataset(val_df, tokenizer)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'BartTokenizerFast'.


평가지표

In [ ]:
from sklearn.metrics import f1_score, recall_score, roc_auc_score, precision_score
import numpy as np
import torch # torch import 추가

def compute_metrics(eval_pred):
    logits_tuple, labels = eval_pred

    # ✅ logits가 (logits, ...) 구조일 경우 대응
    logits = logits_tuple[0] if isinstance(logits_tuple, (tuple, list)) else logits_tuple

    # ✅ softmax 적용
    softmax_probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    preds = np.argmax(softmax_probs, axis=1)

    # ✅ 안전성 고려: label이 단일 클래스일 경우 예외 처리
    try:
        auc = roc_auc_score(labels, softmax_probs[:, 1])
    except:
        auc = float('nan')

    return {
        "f1": f1_score(labels, preds, average='binary', zero_division=0),
        "recall": recall_score(labels, preds, average='binary', zero_division=0),
        "precision": precision_score(labels, preds, average='binary', zero_division=0),
        "auc_roc": auc
    }


모델 및 학습 설정

In [ ]:
wandb.init(project="kobart-grooming-detect")

model = BartForSequenceClassification.from_pretrained("gogamza/kobart-base-v2", num_labels=2)

training_args = TrainingArguments(
    output_dir="./kobart-grooming-checkpoints",
    eval_strategy="steps",
    eval_steps=10,
    logging_steps=10,
    save_steps=500,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="wandb",  # wandb 사용
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
Some weights of BartForSequenceClassification were not initialized from the model checkpoint at gogamza/kobart-base-v2 and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


스페셜 토큰 추가

In [ ]:
special_tokens_dict = {'additional_special_tokens': ['<SPEAKER_A>', '<SPEAKER_B>']}
tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))

BartScaledWordEmbedding(30002, 768, padding_idx=3)

In [ ]:
val_dataset[0]

{'input_ids': tensor([    1,   232,  1700, 30000, 14538,  1700, 30001, 14581, 24652, 19083,
         20815, 13607,  1700,   214,  1700, 30000, 15458, 16942, 11779, 11763,
           262,  1700, 30001, 14055, 14362, 14476, 20815, 13607,  1700, 30000,
         19102, 11696,  1700, 30000, 14054, 15230, 10338, 14927, 13618, 11763,
           262,  1700, 30001, 14960,   243, 17428, 14538, 13756,  8981, 14153,
         14070, 11542, 19422,  1700,  1223,  1223,  1700, 30000,  1700,  1223,
          1223,  1700, 30000, 20498, 14813, 14333, 23172,  9567,  1700, 30001,
         14813, 19545, 12169, 11763,   262,  1700, 30001, 16063, 14158, 11298,
           262,  1700, 30000, 14960,   243, 14813, 19545, 12169, 11763,  1700,
         30000, 14072, 12154, 10784, 11786, 15941, 16942, 14569, 14570, 23078,
         11264, 14446, 27509, 14158, 11298,  1700, 30000, 14054, 20400, 19393,
         14041, 13383, 12013,  1700,   214,  1700, 30001, 14041,   243, 14177,
          9060, 11763,  1700, 30001, 14

In [ ]:
# 스페셜 토큰 확인
print("Special tokens:", tokenizer.special_tokens_map)

# 추가한 스페셜 토큰 ID 확인
print("Token ID for [SPEAKER_A]:", tokenizer.convert_tokens_to_ids("<SPEAKER_A>"))
print("Token ID for [SPEAKER_B]:", tokenizer.convert_tokens_to_ids("<SPEAKER_B>"))
print("Token ID for [bos_token]:", tokenizer.convert_tokens_to_ids("</s>"))
print("Token ID for [eos_token]:", tokenizer.convert_tokens_to_ids("</s>"))

Special tokens: {'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>', 'additional_special_tokens': ['<SPEAKER_A>', '<SPEAKER_B>']}
Token ID for [SPEAKER_A]: 30000
Token ID for [SPEAKER_B]: 30001
Token ID for [bos_token]: 1
Token ID for [eos_token]: 1


In [ ]:
val_dataset[1]

{'input_ids': tensor([    1, 30000, 14054, 16743, 14127, 10872,   243, 14331,   262, 30000,
         22465,   232, 30001, 22465,   232, 30001, 22465,   232, 30001, 22465,
           232, 30001, 14160, 16564, 14043, 30001, 22465,   232, 30000, 24976,
           262, 30000, 20241, 28395,   262, 30001, 20241, 11207, 15440, 14878,
         11734,   262, 30001, 20241, 11207, 16603, 20201, 17057, 30000, 14041,
           243, 14764, 16410,   245, 30001, 20815,   243, 15338, 13656, 13656,
         30001, 14651, 22044, 14030, 15642, 20838, 30001, 15458, 14031, 17275,
           262, 30000, 17571, 12332,  9567, 14543, 30000, 14692, 30001, 22611,
         14476, 14207, 15414, 29629, 30001,  1700,  1223,  1223, 30001, 20611,
         14160, 14631, 22538, 29629, 30000, 20815, 13607, 14543, 30000, 14075,
           243, 14955, 14112, 17350, 11734,   262, 30001, 16046, 15734, 14432,
         16441,  9828,   262, 30000, 14651, 14877, 29134, 13618, 11763, 19178,
         19178, 30001,  1700,  1223,  1

In [ ]:
decoded_text = tokenizer.decode(val_dataset[2]["input_ids"])
print(decoded_text)

</s> 거야? <SPEAKER_B> 만나서 반갑다! <SPEAKER_A> 나도 반가워. <SPEAKER_B> 그냥 채팅 중이야. <SPEAKER_B> 너는? <SPEAKER_A> 오, 나도 그냥 채팅. <SPEAKER_B> 사진 있어? <SPEAKER_A> 응, 너는? <SPEAKER_B> 몇 장 있어. <SPEAKER_A> 너 진짜 귀엽다. <SPEAKER_B> 고마워! <SPEAKER_B> 너 39살 맞지? <SPEAKER_A> 응. <SPEAKER_B> 오, 멋지네. <SPEAKER_A> 사진 더 있어? <SPEAKER_B> 있을지도 모르는데 어디 있는지 모르겠어. <SPEAKER_B> 너는 있어? <SPEAKER_A> 응, 있어. <SPEAKER_B> 멋지다. <SPEAKER_B> 보여줄 수 있어? <SPEAKER_A> 부모님은 네가 이런 데서 채팅하는 거 어떻게 생각하시니? <SPEAKER_B> 우리 엄마는 신경 안 쓰셔. <SPEAKER_A> 오, 진짜 멋지다. <SPEAKER_B> 여기서 자주 수다 떨어? <SPEAKER_A> 가끔 여기저기서. <SPEAKER_B> 오케이. <SPEAKER_A> 어떤 얘기 좋아해? <SPEAKER_B> 뭐든지 상관없어. <SPEAKER_B> 너는? <SPEAKER_A> 팬티 얘기 ᄏᄏ <SPEAKER_B> ᄏᄏ 진짜? <SPEAKER_A> 응 ᄏᄏ <SPEAKER_B> 어떤 스타일 좋아해? <SPEAKER_A> 다 좋아해. <SPEAKER_B> 오케이. <SPEAKER_B> 특별히 좋아하는 스타일 있어? <SPEAKER_A> 보이쇼츠! 너는? <SPEAKER_B> 나는 비키니 스타일 입어. <SPEAKER_A> 오, 사이즈는? <SPEAKER_B> 5야. <SPEAKER_A> 나이스! <SPEAKER_A> 나도 14 사이즈 입어. <SPEAKER_B> 오, 그래? <SPEAKER_A> 응. <SPEAKER_B> 알겠어. <SPEAKER_B> 팬티도 입어? <SPEAKER_A> 응. <SPEAKER_B> 남자들도 

In [ ]:
logits.shape

NameError: name 'logits' is not defined

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss,F1,Recall,Precision,Auc Roc
10,0.437700,0.108582,0.961538,0.951087,0.972222,0.995097
20,0.173400,0.003132,1.000000,1.000000,1.000000,1.000000


KeyboardInterrupt: 

In [ ]:
predictions = trainer.predict(val_dataset)
from sklearn.metrics import classification_report
print(classification_report(predictions.label_ids, np.argmax(predictions.predictions[0], axis=1)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       184
           1       1.00      1.00      1.00       184

    accuracy                           1.00       368
   macro avg       1.00      1.00      1.00       368
weighted avg       1.00      1.00      1.00       368



In [ ]:
model.save_pretrained("./kobart_grooming_best")
tokenizer.save_pretrained("./kobart_grooming_best")

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:394: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'forced_eos_token_id': 1}
  warnings.warn(


('./kobart_grooming_best/tokenizer_config.json',
 './kobart_grooming_best/special_tokens_map.json',
 './kobart_grooming_best/vocab.json',
 './kobart_grooming_best/merges.txt',
 './kobart_grooming_best/added_tokens.json',
 './kobart_grooming_best/tokenizer.json')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/디스부 최종 프로젝트(온라인 그루밍 범죄 탐지)/kobart_result")
tokenizer.save_pretrained("/content/drive/MyDrive/디스부 최종 프로젝트(온라인 그루밍 범죄 탐지)/kobart_result")

NameError: name 'model' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. 설치 & 모델 준비


In [ ]:
# !pip install transformers pandas tqdm --quiet

from transformers import (
    BartTokenizerFast, BartForSequenceClassification,
    BartTokenizer, BartForConditionalGeneration
)
import pandas as pd
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔹 모델 불러오기
# (인코더 - fine-tuned 분류기)
enc_model = BartForSequenceClassification.from_pretrained("/content/drive/MyDrive/디스부 최종 프로젝트(온라인 그루밍 범죄 탐지)/kobart_result").to(device)
enc_tokenizer = BartTokenizerFast.from_pretrained("/content/drive/MyDrive/디스부 최종 프로젝트(온라인 그루밍 범죄 탐지)/kobart_result")

# (디코더 - 고정 생성기)
# dec_model = BartForConditionalGeneration.from_pretrained("gogamza/kobart-base-v2").to(device)
# dec_tokenizer = BartTokenizerFast.from_pretrained("gogamza/kobart-base-v2")

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.


In [ ]:
import torch.nn.functional as F

# 누적 대화 히스토리
history = []

def predict_grooming_proba(text, model, tokenizer, max_len=512):
    model.eval()

    # ✅ 반드시 </s> 붙이기
    if not text.strip().endswith("</s>"):
        text = text.strip() + " </s>"

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_len,
        padding='max_length',
        add_special_tokens=False  # </s> 직접 처리
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
        pred = probs.argmax()

    return {
        "predicted_label": int(pred),
        "probabilities": {
            "0 (일반 대화)": float(probs[0]),
            "1 (그루밍 의심)": float(probs[1])
        }
    }

In [ ]:
# 대화 루프

while True:
    user_input = input("💬 대화를 입력하세요 (exit 입력 시 종료): ").strip()
    if user_input.lower() == "exit":
        history.clear()
        break

    # ✅ 히스토리에 저장 (최대 30개 유지)
    history.append(user_input)
    if len(history) > 30:
        history.pop(0)  # 가장 오래된 발화 제거

    # ✅ 누적된 대화를 하나의 입력으로 구성
    full_text = " ".join(history) + " </s>"

    # ✅ 예측
    result = predict_grooming_proba(full_text, enc_model, enc_tokenizer)
    for i, conversation in enumerate(history):
        print(f"[대화 히스토리] ({i}) : {conversation}")
    print('=' * 50)
    print(f"[예측 라벨] ➤ {result['predicted_label']}")
    print("[확률 분포]")
    for k, v in result["probabilities"].items():
        print(f" - {k}: {v:.4f}")
    print('=' * 50)

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


[대화 히스토리] (0) : 뭐해
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.7530
 - 1 (그루밍 의심): 0.2470
[대화 히스토리] (0) : 뭐해
[대화 히스토리] (1) : 어디 살아?
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.7545
 - 1 (그루밍 의심): 0.2455
[대화 히스토리] (0) : 뭐해
[대화 히스토리] (1) : 어디 살아?
[대화 히스토리] (2) : 집이야?
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.6465
 - 1 (그루밍 의심): 0.3535
[대화 히스토리] (0) : 뭐해
[대화 히스토리] (1) : 어디 살아?
[대화 히스토리] (2) : 집이야?
[대화 히스토리] (3) : 엄마 집에 계셔?
[예측 라벨] ➤ 1
[확률 분포]
 - 0 (일반 대화): 0.4930
 - 1 (그루밍 의심): 0.5070
[대화 히스토리] (0) : 뭐해
[대화 히스토리] (1) : 어디 살아?
[대화 히스토리] (2) : 집이야?
[대화 히스토리] (3) : 엄마 집에 계셔?
[대화 히스토리] (4) : 사진 보내줄래?
[예측 라벨] ➤ 1
[확률 분포]
 - 0 (일반 대화): 0.1871
 - 1 (그루밍 의심): 0.8129


In [ ]:
history

[]